In [1]:
from typing import Any, Callable, Tuple, Dict
import functools
import inspect
import asyncio
import nest_asyncio
import time

# This library patches asyncio to allow nested event loops
nest_asyncio.apply()
def tool(f):
    """Marks a function as a tool for LLM agent."""

    @functools.wraps(f)
    def wrapper(*args, **kwargs):
        # update globals with local context
        old_context = {}
        if(wrapper.__local_context):
            f.__globals__.update(wrapper.__local_context)
            old_context = {key: f.__globals__.get(key) for key in wrapper.__local_context.keys()}
        result = None
        if inspect.iscoroutinefunction(f):
            # If f is async, run it using the event loop
            result = asyncio.get_running_loop().run_until_complete(f(*args, **kwargs))
        else:
            # If f is sync, just call it
            result = f(*args, **kwargs)
        #restore context
        for key, old_value in old_context.items():
            f.__globals__[key] = old_value
    
    def _set_local_context_dict(locals_dict: Dict):
        """Set locals dict for the wrapped function.
        
        Args:
            locals_dict: Dictionary of locals variables that will be passed to exec. 
                         It can be modiefied durign call of exec()
        """
        wrapper.__local_context = locals_dict

    wrapper.__local_context = None
    wrapper._set_local_context_dict = _set_local_context_dict
    return wrapper

In [2]:
@tool
def function():
    print(f"MESSGAE FROM FUNCT: {message}")
    print(f"MESSAGE FROM FUNCT LOCAL CONTEXT: {local_message}")

@tool
async def async_function():
    print(f"MESSGE FROM ASYNC FUNCT: {message}")
    print(f"MESSAGE FROM ASYNC FUNCT LOCAL CONTEXT: {local_message}")

In [3]:
local_context = {
    "message": "Hello world",
    "function":function,
    "async_function":async_function
}
global_context = globals()

In [4]:
import ast
import sys
import traceback
import textwrap
code = """print(message)
local_message="local message"
function()
async_function()"""
tree = ast.parse(code)
for node in tree.body:
    wrapper = ast.Module(body=[node], type_ignores=[])
    ast.fix_missing_locations(wrapper)
    
    try:
        code_obj = compile(wrapper, filename="<string>", mode="exec")
        exec(code_obj, global_context, local_context)
    except Exception as ex:
        print(f"Exception {ex}")
print(local_context)

Hello world
Exception name 'message' is not defined
Exception name 'message' is not defined
{'message': 'Hello world', 'function': <function function at 0x77ff92a04360>, 'async_function': <function async_function at 0x77ff92a04540>, 'local_message': 'local message'}


In [5]:
local_context = {
    "message": "Hello world",
    "function":function,
    "async_function":async_function
}
function._set_local_context_dict(local_context)
async_function._set_local_context_dict(local_context)

In [6]:
code = """
local_message="local message"
print(message)
function()
async_function()"""
tree = ast.parse(code)
for node in tree.body:
    wrapper = ast.Module(body=[node], type_ignores=[])
    ast.fix_missing_locations(wrapper)
    
    try:
        code_obj = compile(wrapper, filename="<string>", mode="exec")
        exec(code_obj, global_context, local_context)
    except Exception as ex:
        print(f"Exception {ex}")

Hello world
MESSGAE FROM FUNCT: Hello world
MESSAGE FROM FUNCT LOCAL CONTEXT: local message
MESSGE FROM ASYNC FUNCT: Hello world
MESSAGE FROM ASYNC FUNCT LOCAL CONTEXT: local message
